In [0]:


from pyspark.sql import functions as F

yellow0 = (spark.table("yellow_trips_raw")
             .withColumnRenamed("tpep_pickup_datetime",  "pickup_datetime")
             .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
             .withColumn("service_type", F.lit("yellow")))

green0  = (spark.table("green_trips_raw")
             .withColumnRenamed("lpep_pickup_datetime",  "pickup_datetime")
             .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
             .withColumn("service_type", F.lit("green")))

def fmt(n: int) -> str:
    return f"{n:,}"

print("RAW rows — YELLOW:", fmt(yellow0.count()))
print("RAW rows — GREEN :", fmt(green0.count()))


RAW rows — YELLOW: 907,982,776
RAW rows — GREEN : 83,484,688


In [0]:


zones = (spark.table("workspace.bde.taxi_zone_lookup")
           .select(F.col("LocationID").alias("locid"),
                   F.col("Borough").alias("borough")))

NYC5 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

def with_features(df):
    df = (df
          .withColumn("trip_distance", F.col("trip_distance").cast("double"))
          .withColumn("duration_sec", F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long"))
          .withColumn("duration_min", F.col("duration_sec")/60.0)
          .withColumn("duration_hr",  F.col("duration_sec")/3600.0)
          .withColumn("speed_mph",    F.when(F.col("duration_hr") > 0, F.col("trip_distance")/F.col("duration_hr"))))

    df = (df.alias("t")
          .join(F.broadcast(zones).alias("pu"), F.col("t.pulocationid")==F.col("pu.locid"), "left")
          .join(F.broadcast(zones).alias("do"), F.col("t.dolocationid")==F.col("do.locid"), "left")
          .withColumn(
              "in_nyc",
              F.col("pu.borough").isin(NYC5) & F.col("do.borough").isin(NYC5)
          )
          # keep ONLY original columns + in_nyc (drop all joined columns)
          .select("t.*", "in_nyc")
         )
    return df

yellow = with_features(yellow0)
green  = with_features(green0)



In [0]:
# Wide time window so I don't drop valid historical trips.
MIN_TS = F.to_timestamp(F.lit("2009-01-01"))
MAX_TS = F.to_timestamp(F.lit("2025-01-01"))

# Duration (minutes/hours)
MIN_DUR_MIN = 1.0
MAX_DUR_HR  = 3.0

# Distance (miles)
MIN_DIST_MI = 0.10
MAX_DIST_MI = 100.0

# Speed caps (mph): stricter in NYC, looser outside (highway).
MAX_SPEED_NYC     = 60.0
MAX_SPEED_OUTSIDE = 85.0


In [0]:
cond_a = F.col("dropoff_datetime") < F.col("pickup_datetime")
green_a, yellow_a = apply_rule(green, yellow, cond_a, "4a", "Trips finishing before the starting time")



Trips finishing before the starting time (4a)
Invalid GREEN trips (4a): 1,042
Green after cleaning (4a): 83,483,646
Invalid YELLOW trips (4a): 94,024
Yellow after cleaning (4a): 907,888,752


In [0]:
cond_b = (~F.col("pickup_datetime").between(MIN_TS, MAX_TS)) | (~F.col("dropoff_datetime").between(MIN_TS, MAX_TS))
green_b, yellow_b = apply_rule(green_a, yellow_a, cond_b, "4b", "Pickup/Dropoff datetime outside allowed range")



Pickup/Dropoff datetime outside allowed range (4b)
Invalid GREEN trips (4b): 145
Green after cleaning (4b): 83,483,501
Invalid YELLOW trips (4b): 2,004
Yellow after cleaning (4b): 907,886,748


In [0]:
cond_c = F.col("speed_mph") < 0
green_c, yellow_c = apply_rule(green_b, yellow_b, cond_c, "4c", "Negative speed")



Negative speed (4c)
Invalid GREEN trips (4c): 19,526
Green after cleaning (4c): 83,396,700
Invalid YELLOW trips (4c): 11,444
Yellow after cleaning (4c): 906,860,348


In [0]:
cond_d = F.when(F.col("in_nyc"), F.col("speed_mph") > MAX_SPEED_NYC) \
          .otherwise(F.col("speed_mph") > MAX_SPEED_OUTSIDE)
green_d, yellow_d = apply_rule(green_c, yellow_c, cond_d, "4d", "Very high speed")



Very high speed (4d)
Invalid GREEN trips (4d): 198,914
Green after cleaning (4d): 83,197,786
Invalid YELLOW trips (4d): 1,001,039
Yellow after cleaning (4d): 905,859,309


In [0]:
cond_e = (F.col("duration_min") < MIN_DUR_MIN) | (F.col("duration_hr") > MAX_DUR_HR)
green_e, yellow_e = apply_rule(green_d, yellow_d, cond_e, "4e", "Duration out of bounds")




Duration out of bounds (4e)
Invalid GREEN trips (4e): 1,592,628
Green after cleaning (4e): 81,605,158
Invalid YELLOW trips (4e): 7,754,667
Yellow after cleaning (4e): 898,104,642


In [0]:
cond_f = (F.col("trip_distance") < MIN_DIST_MI) | (F.col("trip_distance") > MAX_DIST_MI)
green_f, yellow_f = apply_rule(green_e, yellow_e, cond_f, "4f", "Distance out of bounds")



Distance out of bounds (4f)
Invalid GREEN trips (4f): 646,685
Green after cleaning (4f): 80,958,473
Invalid YELLOW trips (4f): 3,140,126
Yellow after cleaning (4f): 894,964,516


In [0]:
cond_g = ((F.col("fare_amount") < 0) | (F.col("total_amount") < 0)) | \
         (F.col("passenger_count").isNotNull() & ((F.col("passenger_count") <= 0) | (F.col("passenger_count") > 8)))
green_g, yellow_g = apply_rule(green_f, yellow_f, cond_g, "4g",
                               "Extra checks (amounts>=0; passenger_count in [1..8] if present)")



Extra checks (amounts>=0; passenger_count in [1..8] if present) (4g)
Invalid GREEN trips (4g): 125,829
Green after cleaning (4g): 80,832,644
Invalid YELLOW trips (4g): 6,954,701
Yellow after cleaning (4g): 888,009,815


In [0]:
green_clean  = green_g.drop("duration_sec","duration_min","duration_hr","in_nyc")
yellow_clean = yellow_g.drop("duration_sec","duration_min","duration_hr","in_nyc")

total_before = green0.count() + yellow0.count()
total_after  = green_clean.count() + yellow_clean.count()
drop_rate    = (total_before - total_after)/total_before if total_before else 0.0

print("\n=== Final Rows After Cleaning ===")
print(f"Green taxi  final rows : {fmt(green_clean.count())}")
print(f"Yellow taxi final rows : {fmt(yellow_clean.count())}")
print(f"TOTAL final rows       : {fmt(total_after)}")
print(f"Overall removed        : {fmt(total_before - total_after)}  ({drop_rate:.2%})")

assert drop_rate <= 0.10, "Cleaning removed >10%. Loosen caps (speed/duration/distance) and re-run."

# Save cleaned tables for reuse
green_clean.write .format("delta").mode("overwrite").saveAsTable("green_trips_clean_proper")
yellow_clean.write.format("delta").mode("overwrite").saveAsTable("yellow_trips_clean_proper")


=== Final Rows After Cleaning ===
Green taxi  final rows : 80,832,644
Yellow taxi final rows : 888,009,815
TOTAL final rows       : 968,842,459
Overall removed        : 22,625,005  (2.28%)


In [0]:
from pyspark.sql import functions as F
fmt = lambda n: f"{n:,}"

def apply_rule_simple(gdf, ydf, cond, code, title):
    bad_g = gdf.where(cond).count(); bad_y = ydf.where(cond).count()
    g2, y2 = gdf.where(~cond), ydf.where(~cond)
    print(f"\n{title} ({code})")
    print(f"Invalid GREEN ({code}):  {fmt(bad_g)}")
    print(f"Green after ({code})  :  {fmt(g2.count())}")
    print(f"Invalid YELLOW ({code}): {fmt(bad_y)}")
    print(f"Yellow after ({code}) :  {fmt(y2.count())}")
    return g2, y2

# start from your post a–g outputs
g_cur, y_cur = green_g, yellow_g

# cast a few columns once (lightweight)
for name in ["g_cur","y_cur"]:
    df = locals()[name]
    for c,t in [("total_amount","double"),("fare_amount","double"),
                ("tip_amount","double"),("trip_distance","double"),
                ("payment_type","int")]:
        if c in df.columns: df = df.withColumn(c, F.col(c).cast(t))
    if "duration_min" not in df.columns:
        df = df.withColumn("duration_min", (F.col("dropoff_datetime").cast("long")-F.col("pickup_datetime").cast("long"))/60.0)
    locals()[name] = df



In [0]:
# Keep legit free/disputed trips; remove only obviously wrong zero/negative totals
cond_E1 = (F.col("total_amount") <= 0) & (~F.col("payment_type").isin(3,4)) & \
          (F.col("duration_min") >= 1) & (F.col("trip_distance") >= 0.5)

g_e1, y_e1 = apply_rule_simple(g_cur, y_cur, cond_E1, "E1",
                               "Zero/negative total but not No-charge(3)/Dispute(4)")



Zero/negative total but not No-charge(3)/Dispute(4) (E1)
Invalid GREEN (E1):  135,992
Green after (E1)  :  80,696,121
Invalid YELLOW (E1): 15,148
Yellow after (E1) :  887,994,667


In [0]:
cond_E2 = ~F.col("vendorid").isin(1,2)
g_e2, y_e2 = apply_rule_simple(g_e1, y_e1, cond_E2, "E2", "VendorID not in {1,2}")



VendorID not in {1,2} (E2)
Invalid GREEN (E2):  107
Green after (E2)  :  80,696,014
Invalid YELLOW (E2): 878,692
Yellow after (E2) :  887,115,975


In [0]:
cond_E3 = ~F.col("ratecodeid").isin(1,2,3,4,5,6)
g_e3, y_e3 = apply_rule_simple(g_e2, y_e2, cond_E3, "E3", "RateCodeID not in 1..6")



RateCodeID not in 1..6 (E3)
Invalid GREEN (E3):  86
Green after (E3)  :  78,866,678
Invalid YELLOW (E3): 752,843
Yellow after (E3) :  877,913,515


In [0]:
from pyspark.sql import functions as F
fmt = lambda n: f"{n:,}"

# Use your latest cleaned frames (you already set these earlier)
green_latest  = globals().get("green_H5",  globals().get("green_g"))
yellow_latest = globals().get("yellow_H5", globals().get("yellow_g"))
assert green_latest is not None and yellow_latest is not None, "Run your cleaning cells first."


In [0]:
g_rows = green_latest.count()
y_rows = yellow_latest.count()
print("Clean GREEN rows :", fmt(g_rows))
print("Clean YELLOW rows:", fmt(y_rows))
print("Clean TOTAL rows :", fmt(g_rows + y_rows))


Clean GREEN rows : 80,742,772
Clean YELLOW rows: 887,119,125
Clean TOTAL rows : 967,861,897


In [0]:
trips_all = yellow_latest.unionByName(green_latest, allowMissingColumns=True)

In [0]:
zones_tbl = "workspace.bde.taxi_zone_lookup" if spark.catalog.tableExists("workspace.bde.taxi_zone_lookup") else "taxi_zone_lookup"
zones = (spark.table(zones_tbl)
         .select(F.col("LocationID").alias("locid"),
                 F.col("Borough").alias("borough"),
                 F.col("Zone").alias("zone"),
                 F.col("service_zone").alias("service_zone")))

final_df = (trips_all.alias("t")
  .join(F.broadcast(zones).alias("pu"), F.col("t.pulocationid")==F.col("pu.locid"), "left")
  .join(F.broadcast(zones).alias("do"), F.col("t.dolocationid")==F.col("do.locid"), "left")
  .select(
      "t.*",
      F.col("pu.borough").alias("pickup_borough"),
      F.col("pu.zone").alias("pickup_zone"),
      F.col("pu.service_zone").alias("pickup_service_zone"),
      F.col("do.borough").alias("dropoff_borough"),
      F.col("do.zone").alias("dropoff_zone"),
      F.col("do.service_zone").alias("dropoff_service_zone")
  ))


In [0]:
final_df.createOrReplaceTempView("tmp_trips_final")

spark.sql("DROP TABLE IF EXISTS workspace.bde.taxi_trips_final")
spark.sql("""
  CREATE TABLE workspace.bde.taxi_trips_final
  USING DELTA
  AS SELECT * FROM tmp_trips_final
""")

print("Saved table: workspace.bde.taxi_trips_final")
print("Final table rows:", f"{(g_rows + y_rows):,}") 

Saved table: workspace.bde.taxi_trips_final
Final table rows: 967,861,897
